In [ ]:
import numpy as np
from typing import Literal



In [ ]:
class Frame:
    def __init__(self, origin: np.array= np.array([0.0,0.0,0.0]), x: np.array= np.array([1.0,0.0,0.0]), y: np.array= np.array([0.0,1.0,0.0]), z: np.array= None):
        self.origin = origin
        if z is not None:
            self.z = z
            self.z /= np.linalg.norm(self.z)
            ref = np.array([0.0, 1.0, 0.0])
            if abs(np.dot(ref, self.z)) > 0.9:
                ref = np.array([1.0, 0.0, 0.0])
            self.x = np.cross(ref, self.z)
            self.x /= np.linalg.norm(self.x)
            self.y = np.cross(self.z, self.x)
        else:
            self.x = x / np.linalg.norm(x)
            self.y = y - np.dot(y, self.x) * self.x
            self.y /= np.linalg.norm(self.y)
            self.z = np.cross(self.x, self.y)
    @property
    def R(self):
        return np.column_stack([self.x, self.y, self.z])

def project_onto_plane(v, normal:np.array=np.array([0.0,0.0,1.0])):
    n = normal / np.linalg.norm(normal)
    return v - np.dot(v, n) * n

def project_to_frame(p, frame):
    return (p - frame.origin) @ frame.R

def from_frame(p, frame):
    return frame.origin + p @ frame.R.T

def ccw_angle(v, normal=np.array([0.0, 0.0, 1.0])):
    vp = project_onto_plane(v,normal)
    ref = np.array([1.0, 0.0, 0.0])
    if abs(np.dot(ref, normal)) > 0.99:
        ref = np.array([0.0, 1.0, 0.0])
    ref -= np.dot(ref, normal) * normal
    ref /= np.linalg.norm(ref)
    return np.arctan2(np.dot(normal, np.cross(ref, vp)),np.dot(ref, vp)) % (2 * np.pi)

def ray_intersection(p1, d1, p2, d2):
    A = np.column_stack((d1, -d2))
    b = p2 - p1
    try:
        t1, _ = np.linalg.solve(A, b)
        return p1 + t1 * d1
    except np.linalg.LinAlgError:
        return p1

def offset_per_segments(corners:np.array,offsets:np.array):
    _corners = []
    tangents = np.roll(corners,-1,axis=0)-corners
    tangents = tangents/np.linalg.norm(tangents,axis=1)[:,None]
    offset_dirs = tangents[:,::-1]*[1,-1]
    for n in range(len(corners)):
        n1 = (n+1)%len(corners)
        p1 = corners[n]  + offset_dirs[n]  * offsets[n]
        p2 = corners[n1] + offset_dirs[n1] * offsets[n1]
        if np.dot(tangents[n],tangents[n1]) >0.9:
            p1 = corners[n1]  + offset_dirs[n]  * offsets[n]
            _corners.append(p1)
            _corners.append(p2)
        else:
            _corners.append(ray_intersection(p1, tangents[n],p2, tangents[n1]))
    return(np.stack(_corners))

def solve_layer_joint(layer0,layer1, joint_frame:Frame,joint_type: Literal["miter_joint","butt_joint","double_butt_joint"]="miter_joint"):
    #layer 0
    layer0_edge_centroids = (layer0.assembly.face + np.roll(layer0.assembly.face,-1,0))/2
    layer0_edge_idx = np.argmin(np.linalg.norm(layer0_edge_centroids-joint_frame.origin,axis=1))

    layer0_normal = layer0.assembly.frame.z
    layer0_edge_dir = layer0.assembly.face[(layer0_edge_idx+1)%len(layer0.assembly.face)]-layer0.assembly.face[layer0_edge_idx]
    layer0_edge_dir = layer0_edge_dir/np.linalg.norm(layer0_edge_dir)
    layer0_edge_normal = np.cross(layer0_edge_dir,layer0_normal)

    layer0_edge_point_1_3d = (layer0_edge_centroids[layer0_edge_idx] + layer0_normal * layer0.offset)
    layer0_edge_point_2_3d = (layer0_edge_centroids[layer0_edge_idx] + layer0_normal * (layer0.offset + layer0.thickness))
    layer0_edge_point_1 = project_to_frame(layer0_edge_point_1_3d, joint_frame)
    layer0_edge_point_2 = project_to_frame(layer0_edge_point_2_3d, joint_frame)

    layer0_centroid = np.mean(layer0.assembly.face,0) 
    layer0_moment = np.cross(layer0.assembly.frame.z,joint_frame.origin - layer0_centroid)
    layer0_ccw = np.dot(layer0_moment,joint_frame.z)>0

    n0 =  layer0_edge_normal @ joint_frame.R
    n0 /= np.linalg.norm(n0)

    #layer1
    layer1_edge_centroids = (layer1.assembly.face + np.roll(layer1.assembly.face,-1,0))/2
    layer1_edge_idx = np.argmin(np.linalg.norm(layer1_edge_centroids-joint_frame.origin,axis=1))

    layer1_normal = layer1.assembly.frame.z
    layer1_edge_dir = layer1.assembly.face[(layer1_edge_idx+1)%len(layer1.assembly.face)]-layer1.assembly.face[layer1_edge_idx]
    layer1_edge_dir = layer1_edge_dir/np.linalg.norm(layer1_edge_dir)
    layer1_edge_normal = np.cross(layer1_edge_dir,layer1_normal)

    layer1_edge_point_1_3d = (layer1_edge_centroids[layer1_edge_idx] + layer1_normal * layer1.offset)
    layer1_edge_point_2_3d = (layer1_edge_centroids[layer1_edge_idx] + layer1_normal * (layer1.offset + layer1.thickness))
    layer1_edge_point_1 = project_to_frame(layer1_edge_point_1_3d, joint_frame)
    layer1_edge_point_2 = project_to_frame(layer1_edge_point_2_3d, joint_frame)

    layer1_centroid = np.mean(layer1.assembly.face,0) 
    layer1_moment = np.cross(layer1.assembly.frame.z,joint_frame.origin - layer1_centroid)
    layer1_ccw = np.dot(layer1_moment,joint_frame.z)>0

    n1 = layer1_edge_normal @ joint_frame.R
    n1 /= np.linalg.norm(n1)

    flip = np.arctan2(n0[0] * n1[1] - n0[1] * n1[0],np.dot(n0, n1)) % (2 * np.pi)<np.pi
    if abs(np.dot(n0, n1)) > 0.9:
        joint_type ='miter_joint'
        if abs(np.dot(n0, n1)) > 0.999:
            offset = -layer1.inside_offsets[layer1_edge_idx]
            layer0.inside_offsets[layer0_edge_idx] = offset
            layer0.outside_offsets[layer0_edge_idx] = offset
            return

    I11 = ray_intersection(layer0_edge_point_1[:2], n0[:2], layer1_edge_point_1[:2], n1[:2])
    I22 = ray_intersection(layer0_edge_point_2[:2], n0[:2],layer1_edge_point_2[:2], n1[:2])
    I12 = ray_intersection(layer0_edge_point_1[:2], n0[:2],layer1_edge_point_2[:2], n1[:2])
    I21 = ray_intersection(layer0_edge_point_2[:2], n0[:2], layer1_edge_point_1[:2], n1[:2])
    
    if joint_type =='miter_joint':
        if layer0_ccw == layer1_ccw :
            layer0.inside_offsets[layer0_edge_idx] = np.dot(I12 - layer0_edge_point_1[:2],n0[:2])
            layer0.outside_offsets[layer0_edge_idx] = np.dot(I21 - layer0_edge_point_1[:2],n0[:2])
            
            layer1.inside_offsets[layer1_edge_idx] = np.dot(I21 - layer1_edge_point_1[:2],n1[:2])
            layer1.outside_offsets[layer1_edge_idx]  = np.dot(I12 - layer1_edge_point_1[:2],n1[:2])
        else:
            layer0.inside_offsets[layer0_edge_idx] = np.dot(I11 - layer0_edge_point_1[:2],n0[:2])
            layer0.outside_offsets[layer0_edge_idx] = np.dot(I22 - layer0_edge_point_1[:2],n0[:2])
            
            layer1.inside_offsets[layer1_edge_idx] = np.dot(I11 - layer1_edge_point_1[:2],n1[:2])
            layer1.outside_offsets[layer1_edge_idx]  = np.dot(I22 - layer1_edge_point_1[:2],n1[:2])
    
    elif joint_type =='butt_joint':
        if layer0_ccw == layer1_ccw:
            layer0.inside_offsets[layer0_edge_idx] = np.dot(I11 - layer0_edge_point_1[:2],n0[:2])
            layer0.outside_offsets[layer0_edge_idx] = np.dot(I21 - layer0_edge_point_1[:2],n0[:2])
        else:
            if flip:
                layer0_ccw = not layer0_ccw
                layer1_ccw = not layer1_ccw
            if (not layer0_ccw) and layer1_ccw:
                layer0.inside_offsets[layer0_edge_idx] = np.dot(I22 - layer0_edge_point_1[:2],n0[:2])
                layer0.outside_offsets[layer0_edge_idx] = np.dot(I12 - layer0_edge_point_1[:2],n0[:2])    
            else:
                layer0.inside_offsets[layer0_edge_idx] = np.dot(I11 - layer0_edge_point_1[:2],n0[:2])
                layer0.outside_offsets[layer0_edge_idx] = np.dot(I21 - layer0_edge_point_1[:2],n0[:2])    

    elif joint_type =='double_butt_joint':
        if layer0_ccw == layer1_ccw:

            layer0.inside_offsets[layer0_edge_idx] = np.dot(I12 - layer0_edge_point_1[:2],n0[:2])
            layer0.outside_offsets[layer0_edge_idx] = np.dot(I22 - layer0_edge_point_1[:2],n0[:2])

            layer1.inside_offsets[layer1_edge_idx] = np.dot(I21 - layer1_edge_point_1[:2],n1[:2])
            layer1.outside_offsets[layer1_edge_idx] = np.dot(I22 - layer1_edge_point_1[:2],n1[:2])
        else:
            if flip:
                layer0_ccw = not layer0_ccw
                layer1_ccw = not layer1_ccw
            if (not layer0_ccw) and layer1_ccw:
                layer0.inside_offsets[layer0_edge_idx] = np.dot(I11 - layer0_edge_point_1[:2],n0[:2])
                layer0.outside_offsets[layer0_edge_idx] = np.dot(I21 - layer0_edge_point_1[:2],n0[:2]) 

                layer1.inside_offsets[layer1_edge_idx] = np.dot(I21 - layer1_edge_point_1[:2],n1[:2])
                layer1.outside_offsets[layer1_edge_idx] = np.dot(I22 - layer1_edge_point_1[:2],n1[:2])   
            else:
                layer0.inside_offsets[layer0_edge_idx] = np.dot(I12 - layer0_edge_point_1[:2],n0[:2])
                layer0.outside_offsets[layer0_edge_idx] = np.dot(I22 - layer0_edge_point_1[:2],n0[:2])  

                layer1.inside_offsets[layer1_edge_idx] = np.dot(I11 - layer1_edge_point_1[:2],n1[:2])
                layer1.outside_offsets[layer1_edge_idx] = np.dot(I12 - layer1_edge_point_1[:2],n1[:2])     
    
class LayeredAssembly:
    def __init__(self,face:np.array, layers:["ConstructionLayer"]=None,name = 'unnamed',apertures:[np.array]=None):
        self.layers = [] if layers == None else layers
        self.name = name
        self.face = face
        self.apertures = [] if apertures is None else apertures
        self.normal = np.cross(face[1]-face[0],face[0]-face[-1])
        self.frame = Frame(face[0],face[1]-face[0],np.cross(self.normal,face[1]-face[0]))
        
    @property
    def axis(self):
        return(self.frame.x)
    @property
    def origin(self):
        return(self.frame.origin)
    @property
    def priorities(self):
        return([layer.priority for layer in self.layers])

class ConstructionLayer:
    def __init__(self,assembly:LayeredAssembly ,offset,thickness,priority,name='unnamed',joint_type: Literal["miter_joint","butt_joint"]="miter_joint"):
        self.assembly = assembly
        self.offset = offset
        self.thickness = thickness
        self.priority = priority
        self.joint_type = joint_type
        self.name = name
        self.inside_offsets = np.zeros(len(assembly.face))
        self.outside_offsets = np.zeros(len(assembly.face))
    @property
    def axis(self):
        return(self.assembly.frame.x)
    @property
    def origin(self):
        return(self.assembly.frame.origin)
    @property
    def inside_face(self,local_coords=True):
        if local_coords:
            inside_face = project_to_frame(self.assembly.face,self.assembly.frame)[:,:2]
            inside_face = offset_per_segments(inside_face,self.inside_offsets)
            inside_face = np.concat([inside_face,np.zeros_like(inside_face[:,:1])],axis=1)
        return(inside_face)
    @property
    def outside_face(self,local_coords=True):
        if local_coords:
            outside_face = project_to_frame(self.assembly.face,self.assembly.frame)[:,:2]
            outside_face = offset_per_segments(outside_face,self.outside_offsets)
            outside_face = np.concat([outside_face,np.zeros_like(outside_face[:,:1])],axis=1)
        return(outside_face)

def solve_joint(assemblies:[LayeredAssembly],joint_frame:Frame):
    if len(assemblies)<2:
        return
    dirs = []
    is_ccw = []

    for assembly in assemblies:
        centroid = np.mean(assembly.face,0) 
        moment = np.cross(assembly.frame.z,joint_frame.origin - centroid)
        v = np.cross(assembly.frame.z,joint_frame.z)
        if np.dot(joint_frame.z,moment) <0:
            is_ccw.append(False)
            v*=-1
        else:
            is_ccw.append(True)
        dirs.append(v)
    
    ccw_order = np.argsort([ccw_angle(v,joint_frame.z) for v in dirs])
    
    dirs = np.stack(dirs)[ccw_order]
    D = np.clip(dirs @ dirs.T, -1.0, 1.0)
    A = np.arccos(D)
    n = len(dirs)
    avg_angles = (A.sum(axis=1) - np.diag(A)) / (n - 1)
    
    shift = -np.argmin(avg_angles)
    ccw_order = np.roll(ccw_order, shift)
    
    assemblies = [assemblies[i] for i in ccw_order]
    is_ccw = [is_ccw[i] for i in ccw_order]
    num_assemblies = len(assemblies)
    
    ###map priorities
    priorities = {}
    priorities_indices = {}
    for n,assembly in enumerate(assemblies):
        layer_priorities = []
        for layer in assembly.layers:
            layer_priorities.append(layer.priority)
            if layer.priority in priorities:
                priorities[layer.priority].append(layer)
                priorities_indices[layer.priority].append(n)
            else:
                priorities[layer.priority] = [layer]
                priorities_indices[layer.priority] = [n]

    layer_joints = {}
    covered = {}
    for priority in reversed(sorted(priorities.keys())):
        
        layers = priorities[priority]
        indices = priorities_indices[priority]
        for index,layer in zip(indices,layers):
            scan_cw = False
            layer_index_in_assembly = layer.assembly.layers.index(layer)
            if len(layer.assembly.priorities[layer_index_in_assembly:])>0:
                if sorted(layer.assembly.priorities[layer_index_in_assembly:])[-1] > priority:
                    scan_cw = True      
            candidate_index = index
            ###max number of connection per joint
            for _ in range(4):
                if layer in layer_joints.values() or layer in layer_joints.keys():
                    break

                if (not is_ccw[index] or scan_cw) and not (not is_ccw[index] and scan_cw):
                    #go clockwise
                    candidate_index -= 1
                else:
                    #go counterclockwise
                    candidate_index += 1

                candidate_index = (candidate_index)%num_assemblies
                candidate_assembly = assemblies[candidate_index]                

                flip_layers = scan_cw
                if is_ccw[index] and not is_ccw[candidate_index]:
                    flip_layers = not scan_cw
                if not is_ccw[index] and is_ccw[candidate_index]:
                    flip_layers = not scan_cw
                        
                candidate_priorities = (reversed(candidate_assembly.priorities) if flip_layers else candidate_assembly.priorities)
                candidate_layers = (reversed(candidate_assembly.layers) if flip_layers else candidate_assembly.layers)
                
                for candidate_layer, candidate_priority in zip(candidate_layers, candidate_priorities):
                    if candidate_priority > priority:
                        solve_layer_joint(layer,candidate_layer,joint_frame,'butt_joint')
                        layer_joints[layer] = candidate_layer
                        #print(f'higher_priority butt joint {index} {layer.assembly.layers.index(layer)} {layer.name},{candidate_index} {candidate_layer.assembly.layers.index(candidate_layer)} {candidate_layer.name}')
                        break
                    elif int(candidate_priority) == int(priority):
                        if candidate_layer in layer_joints.keys() or candidate_layer in layer_joints.values():
                            if candidate_layer in covered.keys():
                                candidate_layer = covered[candidate_layer]
                                candidate_index = layers.index(candidate_layer)
                            solve_layer_joint(layer,candidate_layer,joint_frame,'butt_joint')
                            layer_joints[layer] = candidate_layer
                            #print(f'visited butt joint {index} {layer.assembly.layers.index(layer)} {layer.name},{candidate_index} {candidate_layer.assembly.layers.index(candidate_layer)} {candidate_layer.name}')
                            break
                        else:
                            if layer.joint_type == 'miter_joint' and candidate_layer.joint_type == 'miter_joint':
                                solve_layer_joint(layer,candidate_layer,joint_frame,'miter_joint')
                                layer_joints[layer] = candidate_layer
                                #print(f'miter joint {index} {layer.assembly.layers.index(layer)} {layer.name},{candidate_index} {candidate_layer.assembly.layers.index(candidate_layer)} {candidate_layer.name} ')
                                break
                            else:
                                solve_layer_joint(layer,candidate_layer,joint_frame,'double_butt_joint')
                                layer_joints[layer] = candidate_layer
                                covered[candidate_layer] = layer
                                #print(f'double butt joint {index} {layer.assembly.layers.index(layer)} {layer.name},{candidate_index} {candidate_layer.assembly.layers.index(candidate_layer)} {candidate_layer.name} ')
                                break
    return(layer_joints)